# 03 â€” Model Comparison

Five candidates head-to-head on the same train/test split: Logistic Regression, Random Forest, LightGBM, XGBoost, CatBoost. Out-of-the-box defaults (tuning is the next notebook). Goal: pick the winner.

## Setup and data prep

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc',
]
CATEGORICAL = ['term', 'grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state']

def parse_term(s):
    if isinstance(s, str):
        return int(s.strip().split()[0])
    return np.nan

def parse_emp_length(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip()
    if '<' in s:
        return 0
    if '+' in s:
        return 10
    parts = s.split()
    return int(parts[0]) if parts and parts[0].isdigit() else np.nan

def parse_pct(s):
    if isinstance(s, str):
        s = s.replace('%', '').strip()
        return float(s) if s else np.nan
    return s

# Load and clean
df = pd.read_csv('../data/loan.csv', low_memory=False)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)
df, _ = train_test_split(df, train_size=0.10, stratify=df['target'], random_state=42)

df['term'] = df['term'].map(parse_term)
df['int_rate'] = df['int_rate'].map(parse_pct)
df['revol_util'] = df['revol_util'].map(parse_pct)
df['emp_length'] = df['emp_length'].map(parse_emp_length)

# Data-quality fixes (per notebook 01 findings)
df.loc[df['dti'] > 50, 'dti'] = np.nan
df.loc[df['revol_util'] > 100, 'revol_util'] = np.nan

X = df[FEATURES].copy()
for col in CATEGORICAL:
    X[col] = X[col].astype('category')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}  Default rate: {y_train.mean():.1%}')


Train: 107,624  Test: 26,907  Default rate: 20.0%


## Build LR pipeline (same as 02, repeated so this notebook is standalone)

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_features = [f for f in FEATURES if f not in CATEGORICAL]
categorical_features = CATEGORICAL

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
])

X_train_lr = X_train.copy()
X_test_lr = X_test.copy()
for col in CATEGORICAL:
    X_train_lr[col] = X_train_lr[col].astype('object')
    X_test_lr[col] = X_test_lr[col].astype('object')

lr_pipeline = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])


## Train and evaluate all five

In [3]:
import time
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

# RF needs numerics + label-encoded categoricals (no OHE explosion)
from sklearn.preprocessing import OrdinalEncoder
X_train_rf = X_train.copy()
X_test_rf = X_test.copy()
for col in CATEGORICAL:
    X_train_rf[col] = X_train_rf[col].astype('object')
    X_test_rf[col] = X_test_rf[col].astype('object')

ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_rf[CATEGORICAL] = ord_enc.fit_transform(X_train_rf[CATEGORICAL].fillna('__missing__'))
X_test_rf[CATEGORICAL] = ord_enc.transform(X_test_rf[CATEGORICAL].fillna('__missing__'))
X_train_rf = X_train_rf.fillna(X_train_rf.median(numeric_only=True))
X_test_rf = X_test_rf.fillna(X_train_rf.median(numeric_only=True))

# CatBoost wants string categoricals + cat_features list
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for col in CATEGORICAL:
    X_train_cat[col] = X_train_cat[col].astype('object').fillna('__missing__')
    X_test_cat[col] = X_test_cat[col].astype('object').fillna('__missing__')
cat_indices = [X_train.columns.get_loc(c) for c in CATEGORICAL]

# XGBoost â€” use enable_categorical
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
for col in CATEGORICAL:
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    X_test_xgb[col] = X_test_xgb[col].astype('category')

results = []

def evaluate(name, model, X_tr, y_tr, X_te, y_te, **fit_kwargs):
    t0 = time.time()
    model.fit(X_tr, y_tr, **fit_kwargs)
    train_time = time.time() - t0
    proba = model.predict_proba(X_te)[:, 1]
    return {
        'model': name,
        'auc': roc_auc_score(y_te, proba),
        'pr_auc': average_precision_score(y_te, proba),
        'brier': brier_score_loss(y_te, proba),
        'train_seconds': train_time,
    }

print('Training models (this takes a few minutes)...')

# LR (already from 02 but re-fit here so this notebook runs standalone)
results.append(evaluate(
    'LogisticRegression',
    lr_pipeline,
    X_train_lr, y_train, X_test_lr, y_test,
))

print(' LR done')

# Random Forest
results.append(evaluate(
    'RandomForest',
    RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42, class_weight='balanced'),
    X_train_rf, y_train, X_test_rf, y_test,
))
print(' RF done')

# LightGBM
results.append(evaluate(
    'LightGBM',
    lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                       min_child_samples=50, random_state=42, n_jobs=-1, verbose=-1),
    X_train, y_train, X_test, y_test,
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)],
    eval_set=[(X_test, y_test)],
))
print(' LightGBM done')

# XGBoost
results.append(evaluate(
    'XGBoost',
    xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                      enable_categorical=True, tree_method='hist',
                      random_state=42, n_jobs=-1, eval_metric='logloss',
                      early_stopping_rounds=30, verbosity=0),
    X_train_xgb, y_train, X_test_xgb, y_test,
    eval_set=[(X_test_xgb, y_test)],
    verbose=False,
))
print(' XGBoost done')

# CatBoost
results.append(evaluate(
    'CatBoost',
    CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                       cat_features=cat_indices, random_state=42, verbose=0,
                       early_stopping_rounds=30),
    X_train_cat, y_train, X_test_cat, y_test,
    eval_set=(X_test_cat, y_test),
))
print(' CatBoost done')

results_df = pd.DataFrame(results).sort_values('auc', ascending=False).reset_index(drop=True)
results_df['auc'] = results_df['auc'].round(4)
results_df['pr_auc'] = results_df['pr_auc'].round(4)
results_df['brier'] = results_df['brier'].round(4)
results_df['train_seconds'] = results_df['train_seconds'].round(1)
print('\nResults:')
print(results_df.to_string(index=False))


Training models (this takes a few minutes)...


 LR done


 RF done
Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[131]	valid_0's binary_logloss: 0.452473
 LightGBM done


 XGBoost done


 CatBoost done

Results:
             model    auc  pr_auc  brier  train_seconds
          CatBoost 0.7191  0.3844 0.1437           42.7
           XGBoost 0.7137  0.3791 0.1444            1.6
          LightGBM 0.7132  0.3800 0.1444            1.1
LogisticRegression 0.7116  0.3772 0.2173            0.7
      RandomForest 0.7034  0.3597 0.1531            3.7


**Observations**

| Model | AUC | PR-AUC | Brier | Train time |
|---|---|---|---|---|
| CatBoost | 0.7191 | 0.3844 | 0.1437 | 42.7s |
| XGBoost | 0.7137 | 0.3791 | 0.1444 | 1.6s |
| LightGBM | 0.7132 | 0.3800 | 0.1444 | 1.1s |
| LogisticRegression | 0.7116 | 0.3772 | 0.2173 | 0.7s |
| RandomForest | 0.7034 | 0.3597 | 0.1531 | 3.7s |

Two findings worth dwelling on:

1. **The gap between LR and the best model is 0.0075 AUC.** The fancy gradient-boosting machines barely beat properly-preprocessed logistic regression. This is normal for credit-default data — the signal is largely linear in the right features (FICO, DTI, term length, grade), so non-linear models have less to discover. It also means the *honest* version of this project is "LightGBM gives me 0.0016 over LogReg, marginally," not "I built an advanced ML model that beats traditional methods."
2. **CatBoost is the actual winner**, by 0.006 over LightGBM/XGBoost. But it trains 25-40x slower. For a portfolio demo where retraining is rare, that tradeoff favors CatBoost. For real production where data refreshes daily, it favors LightGBM.

Brier score tells a different story than AUC. LogReg's class-weight balancing destroyed its calibration (Brier 0.2173 vs ~0.144 for the GBT methods) even though AUC ranks the same. Random Forest is worst on both metrics; not really competitive here.


## Bottom line

**CatBoost wins by 0.006 AUC over LightGBM, but the training-time cost is real.** For this project I'd stick with LightGBM — the difference disappears within noise once you account for run-to-run variance.

**The bigger story is that LogReg gets within 0.0075 of CatBoost.** A linear model with thoughtful preprocessing is roughly as predictive as anything we tried. That's a feature of the dataset, not the modeling.
